# UCS420: Cognitive Computing
## Assignment 4 – A Cognitive FAQ System Using Pandas (Nova 2.0)

In [1]:
import pandas as pd

Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your
own roll number digits as follows


• Take the LAST TWO DIGITS of your roll number.
  For each digit d, compute category = ["billing", "account",
  "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g.
if d%3 gives "account", write a question like “how do I update my registered mobile number”).

• # Example roll number ...23 -> digits 2, 3

• # digit 2 -> category[2 % 3] = general

• # digit 3 -> category[3 % 3] = billing

In [2]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]
categories = ["billing", "account", "general"]

In [ ]:
roll_number = "1024160082"

d1 = int(roll_number[-2])
d2 = int(roll_number[-1])

In [5]:
entry1 = {
    "question": "why was an extra charge added",
    "answer": "Extra charges may be added depending on the selected service.",
    "keywords": "extra charge billing",
    "category": categories[d1 % 3]
}

entry2 = {
    "question": "how do i update my mobile number",
    "answer": "You can update your mobile number from account settings.",
    "keywords": "mobile update account",
    "category": categories[d2 % 3]
}

In [6]:
faq_entries = fixed_entries + [entry1, entry2]
df = pd.DataFrame(faq_entries)
print(df)

                           question  \
0            what is the annual fee   
1             how to reset password   
2       what are your working hours   
3             how can i pay the fee   
4     why was an extra charge added   
5  how do i update my mobile number   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4  Extra charges may be added depending on the se...    extra charge billing   
5  You can update your mobile number from account...   mobile update account   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  account  


Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching
entries ranked by confidence

In [7]:
def score_query(query, df):
    query_words = query.lower().split()

    results = []

    for i in range(len(df)):
        keywords = df.loc[i, "keywords"].lower().split()

        score = 0

        for word in query_words:
            if word in keywords:
                score += 1

        if score > 0:
            results.append((score, i))

    results.sort(reverse=True)

    for score, index in results:
        print("Question:", df.loc[index, "question"])
        print("Answer:", df.loc[index, "answer"])
        print("Confidence Score:", score)
        print()


query = input("Enter your question: ")
score_query(query, df)

Enter your question: fee payment
Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Confidence Score: 2

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Confidence Score: 1



Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it
using the category of one of the personalized entries from Q1, and print the result.

In [8]:
def same_category(category_name, df):
    result = df[df["category"] == category_name]
    return result["question"]


category_name = entry1["category"]

print("Category:", category_name)
print(same_category(category_name, df))

Category: billing
0           what is the annual fee
3            how can i pay the fee
4    why was an extra charge added
Name: question, dtype: object


Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and
save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [9]:
index = 0

new_keyword = input("Enter a new keyword: ")

df.loc[index, "keywords"] = df.loc[index, "keywords"] + " " + new_keyword

file_name = roll_number + "_faq_data.csv"

df.to_csv(file_name, index=False)

print("Updated DataFrame:")
print(df)

print("\nFile saved as:", file_name)

Enter a new keyword: 0
Updated DataFrame:
                           question  \
0            what is the annual fee   
1             how to reset password   
2       what are your working hours   
3             how can i pay the fee   
4     why was an extra charge added   
5  how do i update my mobile number   

                                              answer                 keywords  \
0                          The annual fee is Rs 500.  fee cost price charge 0   
1                   Go to Settings > Reset Password.     password reset login   
2                          We are open 9 AM to 5 PM.   hours timing open time   
3         You can pay via UPI, card, or net banking.      pay payment upi fee   
4  Extra charges may be added depending on the se...     extra charge billing   
5  You can update your mobile number from account...    mobile update account   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  billing  
5  account  

File saved as: 102416009

Q5: Using groupby, print how many FAQ entries you have per category.

In [10]:
category_count = df.groupby("category").size()

print("FAQ entries per category:")
print(category_count)

FAQ entries per category:
category
account    2
billing    3
general    1
dtype: int64


Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one
— it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that
produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [11]:
def score_query_tie(query, df):
    query_words = query.lower().split()

    scores = []

    for i in range(len(df)):
        keywords = df.loc[i, "keywords"].lower().split()

        score = 0

        for word in query_words:
            if word in keywords:
                score += 1

        scores.append((score, i))

    highest_score = max(score for score, i in scores)

    if highest_score == 0:
        print("No matching FAQ found")
        return

    print("Best Matching FAQ(s):")

    for score, index in scores:
        if score == highest_score:
            print("\nQuestion:", df.loc[index, "question"])
            print("Answer:", df.loc[index, "answer"])
            print("Confidence Score:", score)


print("Tie Example:")
score_query_tie("fee", df)

print("\nNon-Tie Example:")
score_query_tie("password reset", df)

Tie Example:
Best Matching FAQ(s):

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Confidence Score: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Confidence Score: 1

Non-Tie Example:
Best Matching FAQ(s):

Question: how to reset password
Answer: Go to Settings > Reset Password.
Confidence Score: 2
